# Data Lab 3 — Descriptive Stats & Process Stability
**Certificate 04 · Data Analytics**  |  [← Back to the Lab Hub](../../index.ipynb)

Uses a shared synthetic **factory production log** — one row per part made across 3 presses, 2 shifts, 5 days. pandas, NumPy and matplotlib are pre-installed in Colab, so there is nothing to install.

## Objectives
By the end you will be able to:
- Summarize a variable with mean, std and quartiles.
- Draw mean +/- 3 sigma control limits.
- Flag out-of-control readings and chart a rolling mean.

## Load

In [ ]:
import pandas as pd
DATA = "https://raw.githubusercontent.com/IF-JL/COEFAM-Labs/main/labs/cert04_data_analytics/data/"
df = pd.read_csv(DATA + "production_log.csv", parse_dates=["timestamp"])
df.head()

## 1 · Summary statistics

In [ ]:
print(df["oven_temp_c"].describe(), "\n")
print("Mean & std by machine:")
print(df.groupby("machine")["oven_temp_c"].agg(["mean","std"]).round(2))

## 2 · Control limits — mean +/- 3 sigma
The basis of a control chart: most readings should sit inside +/- 3 standard deviations.

In [ ]:
import matplotlib.pyplot as plt
s = df[df.machine=="Press_A"]["oven_temp_c"]
mu, sd = s.mean(), s.std()
ucl, lcl = mu + 3*sd, mu - 3*sd
plt.figure(figsize=(7,3)); plt.hist(s, bins=40)
for x, c in [(mu,"k"), (ucl,"r"), (lcl,"r")]:
    plt.axvline(x, color=c, linestyle="--")
plt.title(f"Press_A temp — mean {mu:.1f}, UCL {ucl:.1f}, LCL {lcl:.1f}"); plt.show()

## 3 · Flag out-of-control points
A z-score is how many standard deviations a reading is from its machine mean.

In [ ]:
g = df.groupby("machine")["oven_temp_c"]
df["temp_z"] = (df["oven_temp_c"] - g.transform("mean")) / g.transform("std")
ooc = df[df["temp_z"].abs() > 3]
print("Out-of-control temp readings:", len(ooc))
print(ooc.groupby("machine").size())

## 4 · A simple control chart (rolling mean)

In [ ]:
seq = df[df.machine=="Press_C"].reset_index(drop=True)["oven_temp_c"]
plt.figure(figsize=(9,3))
plt.plot(seq.values, alpha=.25, label="reading")
plt.plot(seq.rolling(50).mean().values, label="rolling mean (50)")
plt.legend(); plt.title("Press_C temperature — rolling mean"); plt.show()

## Debrief
- Which machine is least stable (widest spread, most out-of-control points)?
- This is exactly what the SMIP "Process Stability Analyzer" does — now you have built it by hand.